# V3: lineage-safe mixed-domain humanized-AI experiment

Run this notebook in order on a Colab T4 GPU. It freezes GRADTEX before training, uses Beemo only for development/calibration, and writes artifacts to Drive. Do not inspect or evaluate the frozen GRADTEX file until V3 model selection and calibration are complete.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())

In [ ]:
!git clone https://github.com/bonbon1235312/googlecolab-humanize-detector.git /content/humanized-ai-likelihood || true
%cd /content/humanized-ai-likelihood
!git pull
%cd /content/humanized-ai-likelihood/ml
!pip install -q -e .

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
MIXED_DATA_DIR = Path('/content/drive/MyDrive/v3-mixed-humanized-data')
ARTIFACTS_ROOT = Path('/content/drive/MyDrive/humanized-ai-likelihood-v3')
FROZEN_GRADTEX_DIR = Path('/content/drive/MyDrive/v3-frozen-gradtex')

In [ ]:
# Freeze the external benchmark by immutable revision and hash. This does not parse its rows or labels.
from huggingface_hub import hf_hub_download
from humanized_detector.v3_evaluate import freeze_external_benchmark
GRADTEX_REVISION = '553d859da0255d75a39c385c208f7522a2007f53'
hf_hub_download(repo_id='elisabeth-pl-pl/GRADTEX', repo_type='dataset', filename='test.parquet', revision=GRADTEX_REVISION, local_dir=str(FROZEN_GRADTEX_DIR))
manifest = freeze_external_benchmark(FROZEN_GRADTEX_DIR, FROZEN_GRADTEX_DIR / 'manifest.json', dataset='elisabeth-pl-pl/GRADTEX', revision=GRADTEX_REVISION, split='test_c')
print(manifest)
print('Frozen. Do not read test.parquet until V3 selection and calibration are complete.')

In [ ]:
# Build source-labelled, prompt-lineage-safe mixed data. PADBen Task 5 has 16,233 usable rows per class; use 15,000.
!wget -q https://raw.githubusercontent.com/Toloka/beemo/main/dataset.parquet -O /content/beemo.parquet
from datasets import load_dataset
from humanized_detector.beemo import load_beemo_parquet
from humanized_detector.v3_prepare import V3DataConfig, prepare_v3_dataset
padben_records = [dict(row) for row in load_dataset('JonathanZha/PADBen', 'exhaustive-task5', split='train')]
beemo_records = load_beemo_parquet(Path('/content/beemo.parquet'))
report = prepare_v3_dataset(padben_records, beemo_records, MIXED_DATA_DIR, V3DataConfig(padben_samples_per_class=15_000))
print(report)

In [ ]:
# Begin with one pre-registered ablation. Change VARIANT only after recording this run's development metrics.
VARIANT = 'text_attention'  # text_mean, text_attention, structural, fusion_concat, fusion_gated
ARTIFACTS_DIR = ARTIFACTS_ROOT / VARIANT
!python -m humanized_detector.v3_train --data-dir $MIXED_DATA_DIR --artifacts-dir $ARTIFACTS_DIR --variant $VARIANT --epochs 6 --batch-size 64 --lr 3e-5 --weight-decay 0.01 --max-human-fpr 0.05

In [ ]:
import json
metrics = json.loads((ARTIFACTS_DIR / 'development_metrics.json').read_text())
print(metrics)
print('Checkpoint:', ARTIFACTS_DIR / 'model.pt')
print('Feature normalizer:', ARTIFACTS_DIR / 'feature_normalizer.json')